# Load Data

In [7]:
import pandas as pd
import numpy as np
import datetime
pd.set_option('display.max_rows', 500)

In [8]:
dg = pd.read_excel('../../processed_data/02_matched_with_pagetimes/delegator_full_merged.xlsx')

Treatment: Punish - 0; No Punish - 1

Random_order_del: Del first - ; Del second - 

In [9]:
dg['female'] = np.nan
dg.loc[dg['Sex']=='Female', 'female'] = 1
dg.loc[dg['Sex']=='Male', 'female'] = 0

dg['pass_att1'] = 0
dg.loc[dg['test_slider']==207, 'pass_att1'] = 1
dg['pass_att2'] = 0
dg.loc[dg['attention']==0.5, 'pass_att2'] = 1

#weighted average confidence
dg['wa_confidence'] = np.nan
for i in range(dg.shape[0]):
    wa = 0
    for j in range(11):
        wa += (dg['confidence' + str(j)][i]/100)*j
    dg.loc[i, 'wa_confidence'] = wa

#mode confidence
confidence_cols = [f'confidence{i}' for i in range(11)]
max_confidence = dg[confidence_cols].max(axis=1)
mask = dg[confidence_cols].eq(max_confidence, axis=0)
col_to_confidence_num = pd.Series(range(11), index=confidence_cols)

def calculate_mean_round_numbers(row):
    max_rounds = row.index[row].tolist()
    round_numbers = col_to_confidence_num.loc[max_rounds].values
    mean_round = round_numbers.mean()
    return mean_round

dg['highest_confidence'] = mask.apply(calculate_mean_round_numbers, axis=1)

# overconfidence within subject
dg['overconfidence'] = dg['wa_confidence']-dg['participant.overall_score']

#weighted average difficulty
dg['wa_difficulty'] = np.nan
for i in range(dg.shape[0]):
    wa = 0
    for j in range(11):
        wa += (dg['difficulty' + str(j)][i]/100)*j
    dg.loc[i, 'wa_difficulty'] = wa

#overconfidence compared to others
dg['overconfidence_to_others'] = dg['wa_confidence']-dg['wa_difficulty']

#create var for diff in beliefs
dg['bad_good_del'] = dg['belief_del_bad'] - dg['belief_del_good']
dg['bad_good_nodel'] = dg['belief_nodel_bad'] - dg['belief_nodel_good']
dg['nodel_del_good'] = dg['belief_nodel_good'] - dg['belief_del_good']
dg['nodel_del_bad'] = dg['belief_nodel_bad'] - dg['belief_del_bad']

#interactions with treat
dg['nodel_del_good_x_treat'] = dg['nodel_del_good']*dg['treat']
dg['nodel_del_bad_x_treat'] = dg['nodel_del_bad']*dg['treat']

#binary belief
for i in ['belief_del_good', 'belief_del_bad', 'belief_nodel_good', 'belief_nodel_bad']:
    dg[i+'_bin'] = 0
    dg.loc[dg[i]>0, i+'_bin'] = 1

#convert risk task switching point to interval
dg.loc[dg['switching_point']==9999, 'switching_point'] = 110

#for each person calculate how much time they spent on the punishment belief screen
dg['p_time'] = np.nan
for i in range(dg.shape[0]):
    dg.loc[i, 'p_time'] = (datetime.datetime.utcfromtimestamp(dg['Bel'][i]) - datetime.datetime.utcfromtimestamp(dg['PB'][i])).total_seconds()
#two persons in pilot had different order
dg.loc[dg['participant.code']=='up0txcxh', 'p_time'] = (datetime.datetime.utcfromtimestamp(dg['Risk'][i]) - datetime.datetime.utcfromtimestamp(dg['PB'][i])).total_seconds()
dg.loc[dg['participant.code']=='fbckwczw', 'p_time'] = (datetime.datetime.utcfromtimestamp(dg['Risk'][i]) - datetime.datetime.utcfromtimestamp(dg['PB'][i])).total_seconds()

#interaction
dg['treat_x_overall_score'] = dg['treat']*dg['participant.overall_score']

#socio-demographics cleaning
dg['white'] = 1
dg.loc[dg['Ethnicity']!='White/Caucasian', 'white']=0

dg['tech_use_at_work_often'] = 0
dg.loc[dg['Technology use at work']=='more than once a day', 'tech_use_at_work_often'] = 1

dg['leader'] = 0
dg.loc[(dg['Management experience']=="Yes") & (dg['Leadership/position of power/supervisory duties']=="Yes"), 'leader']=1

dg['went_to_uni'] = dg['Highest education level completed'].apply(
    lambda x: 1 if x in [
        'Undergraduate degree (BA/BSc/other)',
        'Graduate degree (MA/MSc/MPhil/other)',
        'Doctorate degree (PhD/other)'
    ] else 0
)

dg['religious'] = 1
dg.loc[dg['Religious affiliation']=="Non Religious (e.g. Agnostic, Atheist, No Religion)", 'religious'] = 0

dg['technology_score'] = 0
# Increase the technology score by 1 for each condition met
dg['technology_score'] += dg['Weekly device usage'].apply(lambda x: 1 if x == 'Multiple times every day' else 0)
dg['technology_score'] += dg['Dating apps'].apply(lambda x: 1 if x == 'Yes' else 0)
dg['technology_score'] += dg['Computer programming'].apply(lambda x: 1 if x == 'Yes' else 0)
dg['technology_score'] += dg['Cryptocurrency'].apply(lambda x: 1 if x == 'Yes' else 0)
dg['technology_score'] += dg['tech_use_at_work_often'].apply(lambda x: 1 if x == 1 else 0)


#true performance as difference
dg['true_performance'] = sum(abs(dg[f'guess_r{i}'] - dg[f'truth_r{i}']) for i in range(1, 11))/10

#interaction between overall_score and treatment
dg['overall_score_x_treat'] = dg['participant.overall_score']*dg['treat']

In [10]:
# rename and reduce to relevant columns

rename_dict = {
    'participant.code': 'code',
    'participant.overall_score': 'overall_score',
    'session.code': 'session_code',
    'test_slider': 'test_slider',
    'guess_r1': 'guess_r1',
    'truth_r1': 'truth_r1',
    'success_r1': 'success_r1',
    'guess_r2': 'guess_r2',
    'truth_r2': 'truth_r2',
    'success_r2': 'success_r2',
    'guess_r3': 'guess_r3',
    'truth_r3': 'truth_r3',
    'success_r3': 'success_r3',
    'guess_r4': 'guess_r4',
    'truth_r4': 'truth_r4',
    'success_r4': 'success_r4',
    'guess_r5': 'guess_r5',
    'truth_r5': 'truth_r5',
    'success_r5': 'success_r5',
    'guess_r6': 'guess_r6',
    'truth_r6': 'truth_r6',
    'success_r6': 'success_r6',
    'guess_r7': 'guess_r7',
    'truth_r7': 'truth_r7',
    'success_r7': 'success_r7',
    'guess_r8': 'guess_r8',
    'truth_r8': 'truth_r8',
    'success_r8': 'success_r8',
    'guess_r9': 'guess_r9',
    'truth_r9': 'truth_r9',
    'success_r9': 'success_r9',
    'guess_r10': 'guess_r10',
    'truth_r10': 'truth_r10',
    'success_r10': 'success_r10',
    'confidence0': 'confidence0',
    'confidence1': 'confidence1',
    'confidence2': 'confidence2',
    'confidence3': 'confidence3',
    'confidence4': 'confidence4',
    'confidence5': 'confidence5',
    'confidence6': 'confidence6',
    'confidence7': 'confidence7',
    'confidence8': 'confidence8',
    'confidence9': 'confidence9',
    'confidence10': 'confidence10',
    'error_confidence': 'error_confidence',
    'difficulty0': 'difficulty0',
    'difficulty1': 'difficulty1',
    'difficulty2': 'difficulty2',
    'difficulty3': 'difficulty3',
    'difficulty4': 'difficulty4',
    'difficulty5': 'difficulty5',
    'difficulty6': 'difficulty6',
    'difficulty7': 'difficulty7',
    'difficulty8': 'difficulty8',
    'difficulty9': 'difficulty9',
    'difficulty10': 'difficulty10',
    'error_difficulty': 'error_difficulty',
    'random_order_del': 'random_order_del',
    'delegation': 'delegation',
    'guess_last': 'guess_last',
    'truth_last': 'truth_last',
    'success_last': 'success_last',
    'confidence_last': 'confidence_last',
    'belief_del_good': 'belief_del_good',
    'belief_del_bad': 'belief_del_bad',
    'belief_nodel_good': 'belief_nodel_good',
    'belief_nodel_bad': 'belief_nodel_bad',
    'belief_del_good_bin': 'belief_del_good_binary',
    'belief_del_bad_bin': 'belief_del_bad_binary',
    'belief_nodel_good_bin': 'belief_nodel_good_binary',
    'belief_nodel_bad_bin': 'belief_nodel_bad_binary',
    'switching_point': 'switching_point',
    'del_reason': 'del_reason',
    'research_about': 'research_about',
    'unclear': 'unclear',
    'attention': 'attention',
    'Time taken': 'time_taken',
    'Total approvals': 'total_approvals',
    'Ethnicity': 'ethnicity',
    'Gender identity': 'gender',
    'Living abroad': 'abroad',
    'Fluent languages': 'languages',
    'Technology use at work': 'tech_at_work',
    'Employment-sector': 'employ_sector',
    'Leadership/position of power/supervisory duties': 'leadership',
    'Management experience': 'management_exp',
    'Highest education level completed': 'education',
    'Body weight': 'body_weight',
    'Religious affiliation': 'religion',
    'Dating apps': 'dating_apps',
    'Hobbies - categories': 'hobbies',
    'Weekly device usage': 'device_usage',
    'Internet enabled products': 'internet_products',
    'Computer programming': 'programming',
    'Socioeconomic status': 'socio_status',
    'Nft experience': 'nft',
    'Cryptocurrency': 'crypto',
    'Personal income (gbp)': 'income',
    'Age': 'age',
    'Sex': 'sex',
    'Ethnicity simplified': 'ethnicity_simple',
    'Country of birth': 'country_birth',
    'Nationality': 'nationality',
    'Language': 'language',
    'Student status': 'student_status',
    'Employment status': 'employ_status',
    'delegation_hypo': 'delegation_hypo',
    'explain_punish_bad': 'explain_punish_bad',
    'explain_punish_good': 'explain_punish_good',
    'pilot': 'pilot',
    'treat': 'treat',
    'female': 'female',
    'pass_att1': 'pass_att1',
    'pass_att2': 'pass_att2',
    'wa_confidence': 'wa_confidence',
    'highest_confidence': 'highest_confidence',
    'overconfidence': 'overconfidence',
    'wa_difficulty': 'wa_difficulty',
    'overconfidence_to_others': 'overconfidence_to_others',
    'bad_good_del': 'bad_good_del',
    'bad_good_nodel': 'bad_good_nodel',
    'nodel_del_good': 'nodel_del_good',
    'nodel_del_bad': 'nodel_del_bad',
    'nodel_del_good_x_treat': 'nodel_del_good_x_treat',
    'nodel_del_bad_x_treat': 'nodel_del_bad_x_treat',
    'p_time': 'punish_bel_time',
    'technology_score': 'technology_score',
    'religious': 'religious',
    'went_to_uni': 'went_to_uni',
    'leader': 'leader',
    'tech_use_at_work_often': 'tech_use_at_work_often',
    'white': 'white',
    'treat_x_overall_score': 'treat_x_overall_score',
    'true_performance': 'true_performance',
    'overall_score_x_treat': 'overall_score_x_treat'
}

dg.rename(columns=rename_dict, inplace=True)
dg = dg[rename_dict.values()]

In [11]:
dg.to_excel('../../processed_data/04_delegator/delegator_cleaned.xlsx', index=False)